# Experiment 5 – Deletion Test

Goal: test whether the words SHAP and LIME blame for a toxic score actually drive it, by deleting them (comprehensiveness) or keeping only them (sufficiency), compared with random words. Runtime: Colab → T4 GPU.

## Cell 1: install, imports and shared helpers

In [ ]:
!pip -q install detoxify shap lime
import importlib.metadata, os, re, numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, torch, shap, lime, sklearn
from matplotlib import colors as mcolors
from detoxify import Detoxify
from lime.lime_text import LimeTextExplainer
from scipy.stats import spearmanr
print("shap", shap.__version__, "| lime", importlib.metadata.version("lime"), "| torch", torch.__version__)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
OUT = "outputs_exp5"; os.makedirs(OUT, exist_ok=True)
BG, FG = "#FFFFFF", "#000000"                     # pink theme (same as Exp 2): white background, black text, pink bars
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
                     "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
                     "axes.edgecolor": FG, "axes.labelcolor": FG, "text.color": FG,
                     "xtick.color": FG, "ytick.color": FG, "legend.labelcolor": FG})
COLORS = {"original": "#D6006F", "unbiased": "#FF8FC4", "err": "#333333"}   # side-by-side pair differs strongly in lightness
ATTR_CMAP = mcolors.LinearSegmentedColormap.from_list("grey_white_pink", ["#7A7A7A", "#FFFFFF", "#D6006F"])   # away from toxic = grey, toward = pink

MODELS = {m: Detoxify(m, device=DEVICE) for m in ["original", "unbiased"]}

def proba(texts, model="original", bs=64):
    """P(toxic) for a list of strings, batched so LIME's thousands of samples don't run out of memory."""
    texts = [str(t) for t in texts]
    out = []
    for i in range(0, len(texts), bs):
        out.extend(np.atleast_1d(MODELS[model].predict(texts[i:i+bs])["toxicity"]))
    return np.array(out, dtype=float)

def logit(texts, model="original"):
    """Explain log-odds, not probability, so SHAP contributions are additive (shap-exercises notebook)."""
    p = np.clip(proba(texts, model), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def lime_fn(model="original"):
    """friday-03 pattern: list of strings -> (n, 2) array of [P(non-toxic), P(toxic)]."""
    def f(texts):
        p = proba(texts, model)
        return np.column_stack([1 - p, p])
    return f

def word_tokenizer(s, return_offsets_mapping=True):
    """Word-level tokens that keep their trailing punctuation. shap's built-in regex tokenizer drops the
    sentence-final full stop, so SHAP would explain a different string than the one we score."""
    spans = [m.span() for m in re.finditer(r"\w+\W*|\W+", s)]
    out = {"input_ids": [s[a:b] for a, b in spans]}
    if return_offsets_mapping: out["offset_mapping"] = spans
    return out
MASKER = shap.maskers.Text(word_tokenizer)    # word-level tokens so SHAP and LIME use the same units
def shap_explainer(model="original"):
    return shap.Explainer(lambda t: logit(t, model), MASKER)

def check_efficiency(sv, texts, model, tol=1e-2):
    """Efficiency/additivity axiom (W4-L01 p4, textbook p48): base value + sum of SHAP values = model output."""
    f = logit(texts, model)
    recon = np.array([float(sv[i].base_values) + sv[i].values.sum() for i in range(len(texts))])
    err = np.abs(recon - f).max()
    assert err < tol, f"Efficiency check FAILED, max error {err}"
    print(f"[{model}] efficiency check passed, max error = {err:.2e}")

def shap_word_weights(e):
    """Collapse SHAP tokens into lower-case words; sum repeated words."""
    w = {}
    for tok, v in zip(e.data, e.values):
        k = re.sub(r"\W+", "", tok).lower()
        if k: w[k] = w.get(k, 0.0) + float(v)
    return w

def lime_word_weights(text, model="original", seed=SEED, num_samples=2000):
    """LIME on one sentence. labels=(1,) so weights refer to the TOXIC class (friday-01 sign trap)."""
    expl = LimeTextExplainer(class_names=["non-toxic", "toxic"], random_state=seed)
    n = len(set(re.findall(r"\w+", text)))
    exp = expl.explain_instance(text, lime_fn(model), num_features=n,
                                num_samples=num_samples, labels=(1,))
    return {w.lower(): v for w, v in exp.as_list(label=1)}, exp

def agreement(sw, lw, k=3):
    """Krishna et al. (2022)-style agreement: rank correlation, top-1 match, top-k overlap.
    Ranks are used because SHAP is in log-odds and LIME is in probability, so raw sizes are not comparable."""
    common = sorted(set(sw) & set(lw))
    rho = spearmanr([sw[w] for w in common], [lw[w] for w in common]).correlation if len(common) > 2 else np.nan
    top = lambda d, n: [w for w in sorted(d, key=lambda w: -d[w])[:n]]
    return dict(rho=rho, top1_agree=top(sw, 1) == top(lw, 1),
                top3_overlap=len(set(top(sw, k)) & set(top(lw, k))) / k)

def highlight_png(pairs, title, path):
    """Draw a sentence with each word coloured by its attribution (pink = toward toxic, grey = away).
    Colours are scaled per figure, so compare colour patterns between SHAP and LIME, not exact shades."""
    vmax = max([abs(v) for _, v in pairs] + [1e-9])
    norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax); cmap = ATTR_CMAP
    fig, ax = plt.subplots(figsize=(14, 1.4)); ax.axis("off")
    fig.canvas.draw(); r = fig.canvas.get_renderer(); x = 0.005
    for w, v in pairs:
        t = ax.text(x, 0.5, w, fontsize=20, va="center", transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.25", fc=cmap(norm(v)), ec="none"))
        x = t.get_window_extent(renderer=r).transformed(ax.transAxes.inverted()).x1 + 0.01
    ax.set_title(title, loc="left", fontsize=16)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)

KS = [1, 2, 3]
N_RANDOM = 10
def toks_of(text): return word_tokenizer(text)["input_ids"]
def key(tok): return re.sub(r"\W+", "", tok).lower()
def ranking(scores):
    """Token positions ordered from most to least 'toward toxic'."""
    return list(np.argsort(-np.asarray(scores, float), kind="stable"))
def drop(toks, keep_idx):
    return "".join(t for j, t in enumerate(toks) if j in keep_idx).strip()

BG, FG = "#FFFFFF", "#000000"                     # pink theme (same as Exp 1 and 2): white background, black text, pink marks
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
                     "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
                     "axes.edgecolor": FG, "axes.labelcolor": FG, "text.color": FG,
                     "xtick.color": FG, "ytick.color": FG, "legend.labelcolor": FG})
COLORS = {"original": "#D6006F", "unbiased": "#FF8FC4", "control": "#9A8590",
          "n500": "#A3134F", "n5000": "#FF7AB8", "err": "#333333",
          "SHAP": "#D6006F", "LIME": "#FF8FC4", "random": "#9A8590",
          "SHAP partition": "#D6006F", "SHAP permutation": "#7A0A3E"}
PINK_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list("pink_black", ["#FFF0F6", "#FF4FA3", "#7A0A3E"])


## Cell 2: the sentences

Upload `exp1_sentences.csv` (from `experiments/report vs promote/data/`) to the Colab session first. The 190 Exp 2 sentences are rebuilt with EXP2 Cell 2's code.

In [ ]:
exp1 = pd.read_csv("exp1_sentences.csv")
TEMPLATES = ["I am a {} person.", "My neighbour is {} and she is lovely.",
             "{} people are welcome here.", "My best friend is {}.",
             "I grew up in a {} family.", "Being {} is completely normal.",
             "We hired a {} engineer last week.", "She is a proud {} woman.",
             "He is {} and works as a teacher.", "Our new {} classmate joined the team."]
TERMS = ["gay","lesbian","transgender","black","white","asian","muslim","jewish",
         "christian","immigrant","deaf","blind","disabled","poor","rural"]
CONTROLS = ["tall","young","local","quiet"]
exp2 = pd.DataFrame([(t.format(w), "control" if w in CONTROLS else "identity") for t in TEMPLATES for w in TERMS + CONTROLS],
                    columns=["text", "group"])
S = pd.concat([pd.DataFrame({"text": exp1.text, "group": exp1.condition, "source": "exp1"}),
               exp2.assign(source="exp2")], ignore_index=True)
print(S.groupby(["source","group"]).size())          # expect 16/16/16 and 150/40

## Cell 3: attributions per token (SHAP and LIME, both models)

SHAP gives a value per token directly. LIME gives a value per word, which we map back onto the tokens.

In [ ]:
ATTR = {}
for m in MODELS:
    sv = shap_explainer(m)(list(S.text), max_evals=500, batch_size=32)
    check_efficiency(sv, list(S.text), m)
    ATTR[(m, "SHAP")] = [list(sv[i].values) for i in range(len(S))]
    lime_scores = []
    for text in S.text:
        lw, _ = lime_word_weights(text, m, seed=SEED, num_samples=5000)
        lime_scores.append([lw.get(key(t), 0.0) for t in toks_of(text)])
    ATTR[(m, "LIME")] = lime_scores
    # sanity: SHAP's tokens are the same as word_tokenizer's
    assert all(len(ATTR[(m, "SHAP")][i]) == len(toks_of(S.text[i])) for i in range(len(S)))

## Cell 4: comprehensiveness and sufficiency

Every perturbed sentence is collected first and then scored in one batch per model (much faster). The last lines print 3 perturbed sentences so you can read them (Part 6 check).

In [ ]:
rng = np.random.RandomState(SEED)
rows = []
for m in MODELS:
    base = logit(S.text, m)
    batch, meta = [], []
    for i, text in enumerate(S.text):
        toks = toks_of(text); n = len(toks)
        for k in KS:
            if k >= n: continue
            sets = {"SHAP": ranking(ATTR[(m, "SHAP")][i])[:k], "LIME": ranking(ATTR[(m, "LIME")][i])[:k]}
            for r in range(N_RANDOM): sets[f"random{r}"] = list(rng.choice(n, k, replace=False))
            for name, top in sets.items():
                top = set(top)
                batch += [drop(toks, set(range(n)) - top), drop(toks, top)]
                meta.append((i, k, name))
    scores = logit(batch, m).reshape(-1, 2)          # column 0 = top-k removed, column 1 = only top-k kept
    for (i, k, name), (removed, kept) in zip(meta, scores):
        rows.append(dict(model=m, i=i, k=k, explainer="random" if name.startswith("random") else name,
                         comp=base[i] - removed, suff=base[i] - kept))
D = pd.DataFrame(rows).groupby(["model","i","k","explainer"], as_index=False)[["comp","suff"]].mean()  # averages the 10 random draws
D = D.merge(S, left_on="i", right_index=True)
D.to_csv(f"{OUT}/exp5_deletion_by_sentence.csv", index=False)

for j in [0, 1, 2]:                                  # Part 6 check: read a few perturbed sentences (last model)
    print(meta[j], "| removed:", repr(batch[2*j]), "| kept only:", repr(batch[2*j+1]))

## Cell 5: AOPC summary (Table I numbers)

AOPC = the average over k = 1, 2, 3. Higher comprehensiveness is better; lower sufficiency is better.

In [ ]:
aopc = D.groupby(["model","explainer"])[["comp","suff"]].mean().round(3)
print(aopc); aopc.to_csv(f"{OUT}/exp5_aopc.csv")
by_group = D.groupby(["model","source","group","explainer"])[["comp","suff"]].mean().round(3)
print(by_group); by_group.to_csv(f"{OUT}/exp5_aopc_by_group.csv")

## Cell 6: does SHAP / LIME beat random in each sentence?

In [ ]:
piv = D.pivot_table(index=["model","i","k"], columns="explainer", values="comp").reset_index()
for m in MODELS:
    p = piv[piv.model == m]
    print(f"[{m}] SHAP beats random on comprehensiveness in {(p.SHAP > p.random).mean():.0%} of cases; "
          f"LIME in {(p.LIME > p.random).mean():.0%}; SHAP beats LIME in {(p.SHAP > p.LIME).mean():.0%}")

## Cell 7: backup figure (not planned for the paper; for the journal and questions)

Comprehensiveness by k for SHAP, LIME and random (original model), one line each.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, g in D[D.model == "original"].groupby("explainer"):
    s = g.groupby("k").comp.mean(); ax.plot(s.index, s.values, marker="o", label=name, color=COLORS[name])
ax.set_xlabel("k (words removed)"); ax.set_ylabel("Comprehensiveness (log-odds drop)")
ax.set_xticks(KS); ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_exp5_comprehensiveness.png", dpi=300)

## Cell 8: results summary

In [ ]:
print("=== EXPERIMENT 5 SUMMARY ===")
print(aopc)
print("Biggest SHAP-over-random gap by group (original):")
g = by_group.xs("original", level="model")["comp"].unstack("explainer")
print((g["SHAP"] - g["random"]).sort_values(ascending=False).round(3))

## Cell 9: download everything

In [ ]:
!zip -qr outputs_exp5.zip outputs_exp5
from google.colab import files; files.download("outputs_exp5.zip")